In [1]:
from pathlib import Path
import os
import sys
import glob

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = (
    SparkSession.builder
    .appName("OlistAnalysis")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

spark

file_paths = glob.glob("../data/raw/*.csv")

dfs = {}
for path in file_paths:
    filename = os.path.splitext(os.path.basename(path))[0]
    clean_name = filename.replace("olist_", "").replace("_dataset", "")
    dfs[clean_name] = spark.read.csv(path, header=True, inferSchema=True, multiLine=True, escape='"')


customers = dfs["customers"]
geolocation = dfs["geolocation"]
order_items = dfs["order_items"]
order_payments = dfs["order_payments"]
order_reviews = dfs["order_reviews"]
orders = dfs["orders"]
products = dfs["products"]
sellers = dfs["sellers"]
product_category_name_translation = dfs["product_category_name_translation"]

In [8]:
customers.printSchema()
print(f"customer count: {customers.count()}")
customers.show(15, truncate=False)

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

customer count: 99441
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city        |customer_state|
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|06b8999e2fba1a1fbc88172c00ba8bc7|861eff4711a542e4b93843c6dd7febb0|14409                   |franca               |SP            |
|18955e83d337fd6b2def6b18a428ac77|290c77bc529b7ac935b93aa66c333dc3|9790                    |sao bernardo do campo|SP            |
|4e7b3e00288586ebd08712fdd0374a03|060e732b5b29e8181a18229c7b0b2b5e|1151          

Customer count is at 99,441 with all columns correctly typed. The customer's state is abbreviated to two letters, while customer's city all lower-case. Customer's zip code prefix seems to have varying lengths but may not be a mistake in Brazil. There is a unique customer id and a customer id which may imply a connection with other tables (such as orders) that will need to be discovered.

In [13]:
customers.select(
    F.count(F.lit(1)).alias("customers_count"),
    F.count(F.col("customer_id")).alias("non_null_customers"),
    F.countDistinct(F.col("customer_id")).alias("n_customer_id_column"),
    F.countDistinct(F.col("customer_unique_id")).alias("n_unique_customer_id_column")
).show()

+---------------+------------------+--------------------+---------------------------+
|customers_count|non_null_customers|n_customer_id_column|n_unique_customer_id_column|
+---------------+------------------+--------------------+---------------------------+
|          99441|             99441|               99441|                      96096|
+---------------+------------------+--------------------+---------------------------+



It appears that customer_id is essentially a per-order customer key. One is created for every order. customer_unique_id is the actual person. Need to find out how many distinct people ordered more than once.

In [41]:
repeat_customers = (
    customers
    .groupBy(F.col("customer_unique_id"))
    .agg(F.count(F.col("customer_unique_id")).alias("n_unique"))
    .where(F.col("n_unique") > F.lit(1))
    .orderBy(F.col("n_unique").desc())
)

print(f"count of customers who are repeat buyers: {repeat_customers.count()}")
repeat_customers.show(truncate=False)


count of customers who are repeat buyers: 2997
+--------------------------------+--------+
|customer_unique_id              |n_unique|
+--------------------------------+--------+
|8d50f5eadf50201ccdcedfb9e2ac8455|17      |
|3e43e6105506432c953e165fb2acf44c|9       |
|1b6c7548a2a1f9037c1fd3ddfed95f33|7       |
|6469f99c1f9dfae7733b25662e7f1782|7       |
|ca77025e7201e3b30c44b472ff346268|7       |
|12f5d6e1cbf93dafd9dcc19095df0b3d|6       |
|dc813062e0fc23409cd255f7f53c7074|6       |
|47c1a3033b8b77b3ab6e109eb4d5fdf3|6       |
|f0e310a6839dce9de1638e0fe5ab282a|6       |
|63cfc61cee11cbe306bff5857d00bfe4|6       |
|de34b16117594161a6a89c50b289d35a|6       |
|4e65032f1f574189fb793bac5a867bbc|5       |
|394ac4de8f3acb14253c177f0e15bc58|5       |
|35ecdf6858edc6427223b64804cf028e|5       |
|74cb1ad7e6d5674325c1f99b5ea30d82|5       |
|fe81bb32c243a86b2f86fbf053fe6140|5       |
|b4e4f24de1e8725b74e4a1f4975116ed|5       |
|56c8638e7c058b98aae6d74d2dd6ea23|5       |
|5e8f38a9a1c023f3db718edcf926

I verified the grain of orders by linking the customer_id from orders to the customer_id in customers. There I saw that there was a unique key that signified unique customers (which I proved with a query in counting number of unique customers). I got the repeat customer count which was 2,997. This number is smaller than the 3,345 number (difference between customer_id and customer_unique_id) because there are several customers that purchased many times as opposed to others.